# 🏥 Medical RAG Testing Pipeline
## Industry-Level Evaluation Framework for Medical Exams (ABR Radiology, NEET, USMLE, etc.)

---

### Overview
This pipeline allows you to:
1. **Upload exam papers** (PDF, DOCX, TXT, JSON) — ABR Radiology, NEET, USMLE, GPQA Diamond, etc.
2. **Upload or configure your RAG model** (HuggingFace `.pt`/`.pth`, API-based, or custom class)
3. **Run structured evaluation** with GPQA Diamond-style metrics
4. **Compare multiple models** across exams
5. **Generate full evaluation reports** (HTML + CSV + JSON)

---

### ⚡ Quick Start
1. Run **Section 0** to install dependencies
2. Run **Section 1** to configure your environment
3. Upload exam paper in **Section 2**
4. Upload/configure your model in **Section 3**
5. Run evaluation in **Section 4**
6. View results in **Section 5**

---

> **Note:** Designed for hospital/clinical AI teams. All data stays local unless you configure an external API judge.

## Section 0: Install Dependencies

In [ ]:
# ============================================================
# SECTION 0: INSTALL ALL REQUIRED PACKAGES
# Run this once. Restart kernel after installation.
# ============================================================

import subprocess, sys

packages = [
    "torch",
    "transformers",
    "sentence-transformers",
    "faiss-cpu",
    "langchain",
    "langchain-community",
    "pypdf",
    "python-docx",
    "pandas",
    "numpy",
    "scikit-learn",
    "matplotlib",
    "seaborn",
    "tqdm",
    "ipywidgets",
    "openai",          # optional: for GPT-based judge
    "anthropic",       # optional: for Claude-based judge
    "rouge-score",
    "nltk",
    "bert-score",
    "pydantic",
    "rich",
    "jinja2",
]

for pkg in packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"✅ {pkg}")
    except Exception as e:
        print(f"⚠️  {pkg} — {e}")

print("\n✅ Done! Restart kernel if this is your first run.")

## Section 1: Configuration & Global Setup

In [ ]:
# ============================================================
# SECTION 1: GLOBAL CONFIGURATION
# Edit these settings before running the pipeline.
# ============================================================

import os
import json
import warnings
import logging
import datetime
import pathlib

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)

# ─── PIPELINE METADATA ────────────────────────────────────────────────────────
PIPELINE_CONFIG = {
    # Your name / institution
    "evaluator_name": "Your Name",
    "institution": "Hospital / Research Institute Name",
    
    # Exam type being tested — used in reports
    # Options: "ABR_Radiology", "NEET", "USMLE_Step1", "USMLE_Step2",
    #           "GPQA_Diamond", "NBME", "custom"
    "exam_type": "ABR_Radiology",
    
    # Model/run label — used to compare runs later
    "model_run_label": "run_v1",
    
    # Output directory for all results
    "output_dir": "./rag_eval_results",
    
    # Maximum questions to evaluate (None = all)
    "max_questions": None,
    
    # Random seed for reproducibility
    "seed": 42,
}

# ─── RETRIEVAL CONFIGURATION ──────────────────────────────────────────────────
RETRIEVAL_CONFIG = {
    # Number of top-k passages retrieved per question
    "top_k": 5,
    
    # Chunk size (tokens) when splitting documents for retrieval
    "chunk_size": 512,
    
    # Chunk overlap
    "chunk_overlap": 64,
    
    # Embedding model for FAISS retrieval
    # Options: "sentence-transformers/all-MiniLM-L6-v2"
    #          "sentence-transformers/all-mpnet-base-v2"
    #          "BAAI/bge-large-en-v1.5"  ← recommended for medical
    #          "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-sst2"  ← biomedical
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    
    # Retrieval strategy: "dense", "bm25", "hybrid"
    "retrieval_strategy": "dense",
}

# ─── JUDGE / EVALUATION LLM CONFIGURATION ────────────────────────────────────
JUDGE_CONFIG = {
    # Which judge to use for LLM-as-judge scoring
    # Options: "gpt-4", "claude-3-opus", "local", "none"
    "judge_model": "none",      # Set to "none" to use only metric-based scoring
    
    # API keys (leave empty to use env vars)
    "openai_api_key": "",       # or set env var OPENAI_API_KEY
    "anthropic_api_key": "",    # or set env var ANTHROPIC_API_KEY
    
    # Medical domain prompt injection for judge
    "judge_system_prompt": """You are an expert medical examiner and board-certified radiologist 
evaluating AI model answers on professional medical examinations. 
Score strictly based on clinical accuracy, safety, and evidence-based medicine.""",
}

# ─── METRICS TO COMPUTE ───────────────────────────────────────────────────────
METRICS_CONFIG = {
    "exact_match": True,          # Exact string match (MCQ)
    "accuracy": True,             # Per-question accuracy
    "rouge": True,                # ROUGE-1, ROUGE-2, ROUGE-L
    "bert_score": True,           # BERTScore (F1)
    "retrieval_precision": True,  # P@k for retrieved passages
    "hallucination_score": True,  # Simple faithfulness check
    "latency": True,              # Time per query
    "llm_judge": False,           # LLM-as-judge (requires API key)
    
    # GPQA Diamond style: double-blind evaluation
    "gpqa_style": True,
}

# ─── SETUP OUTPUT DIR ─────────────────────────────────────────────────────────
output_path = pathlib.Path(PIPELINE_CONFIG["output_dir"])
output_path.mkdir(parents=True, exist_ok=True)

RUN_ID = f"{PIPELINE_CONFIG['model_run_label']}_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = output_path / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Configuration loaded.")
print(f"📁 Results will be saved to: {RUN_DIR.resolve()}")
print(f"🏷️  Run ID: {RUN_ID}")
print(f"🔬 Exam type: {PIPELINE_CONFIG['exam_type']}")

## Section 2: Upload Exam Paper

Upload your exam file here. Supported formats:
- **PDF** — scanned or digital exam papers
- **DOCX** — Word documents
- **JSON** — pre-structured QA pairs
- **TXT** — plain text

**JSON format expected:**
```json
[
  {
    "id": "Q001",
    "question": "A 45-year-old presents with...",
    "options": {"A": "...", "B": "...", "C": "...", "D": "..."},
    "answer": "B",
    "explanation": "...",
    "category": "Thoracic Radiology",
    "difficulty": "hard"
  }
]
```

In [ ]:
# ============================================================
# SECTION 2A: EXAM PAPER UPLOAD (Interactive Widget)
# ============================================================

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import io

# ─── File Upload Widget ────────────────────────────────────────────────────────
exam_uploader = widgets.FileUpload(
    accept='.pdf,.docx,.txt,.json',
    multiple=False,
    description='📄 Upload Exam',
    layout=widgets.Layout(width='400px')
)

exam_status = widgets.Output()

def on_exam_upload(change):
    with exam_status:
        clear_output()
        if exam_uploader.value:
            fname = list(exam_uploader.value.keys())[0]
            fsize = len(list(exam_uploader.value.values())[0]['content'])
            print(f"✅ Uploaded: {fname} ({fsize/1024:.1f} KB)")

exam_uploader.observe(on_exam_upload, names='value')

display(HTML("<h4>Step 1: Upload your exam paper</h4>"))
display(exam_uploader, exam_status)

In [ ]:
# ============================================================
# SECTION 2B: EXAM PAPER PARSER
# Parses uploaded file into structured QA list.
# ============================================================

import re
import json
import tempfile
import pathlib
from typing import List, Dict, Optional, Any

# ─── Data Models ──────────────────────────────────────────────────────────────
from dataclasses import dataclass, field

@dataclass
class ExamQuestion:
    id: str
    question: str
    options: Optional[Dict[str, str]] = None   # {"A": "...", "B": "..."}
    answer: Optional[str] = None               # Correct option key or free-text
    explanation: Optional[str] = None
    category: Optional[str] = None
    difficulty: Optional[str] = None
    source: Optional[str] = None
    metadata: Dict[str, Any] = field(default_factory=dict)

class ExamParser:
    """Parses exam files into structured ExamQuestion lists."""
    
    @staticmethod
    def from_json(data: bytes) -> List[ExamQuestion]:
        raw = json.loads(data.decode("utf-8"))
        questions = []
        for i, item in enumerate(raw):
            q = ExamQuestion(
                id=item.get("id", f"Q{i+1:04d}"),
                question=item["question"],
                options=item.get("options"),
                answer=item.get("answer"),
                explanation=item.get("explanation"),
                category=item.get("category"),
                difficulty=item.get("difficulty"),
                source=item.get("source"),
                metadata={k: v for k, v in item.items() 
                          if k not in ["id","question","options","answer",
                                       "explanation","category","difficulty","source"]}
            )
            questions.append(q)
        return questions

    @staticmethod
    def from_pdf(data: bytes) -> List[ExamQuestion]:
        from pypdf import PdfReader
        reader = PdfReader(io.BytesIO(data))
        full_text = ""
        for page in reader.pages:
            full_text += page.extract_text() + "\n"
        return ExamParser._parse_text_mcq(full_text)

    @staticmethod
    def from_docx(data: bytes) -> List[ExamQuestion]:
        from docx import Document
        doc = Document(io.BytesIO(data))
        full_text = "\n".join([p.text for p in doc.paragraphs])
        return ExamParser._parse_text_mcq(full_text)

    @staticmethod
    def from_txt(data: bytes) -> List[ExamQuestion]:
        full_text = data.decode("utf-8", errors="ignore")
        return ExamParser._parse_text_mcq(full_text)

    @staticmethod
    def _parse_text_mcq(text: str) -> List[ExamQuestion]:
        """
        Heuristic MCQ parser.
        Detects: numbered questions, (A)(B)(C)(D) or A. B. C. D. style options.
        For best results, use JSON format.
        """
        questions = []
        # Split on question numbers: "1.", "Q1.", "Question 1"
        pattern = r"(?:^|\n)(?:Question\s*)?(?:Q[\s\.]?)?\s*(\d+)[\.\.\)]\s+"
        parts = re.split(pattern, text, flags=re.MULTILINE)
        
        q_num = 0
        i = 1
        while i < len(parts) - 1:
            q_id_str = parts[i].strip()
            block = parts[i + 1].strip() if i + 1 < len(parts) else ""
            
            # Try to extract options A-D
            opt_pattern = r"([A-E])[.\)\s]+(.+?)(?=[A-E][.\)\s]|$)"
            opt_matches = re.findall(opt_pattern, block, re.DOTALL)
            options = {k.strip(): v.strip() for k, v in opt_matches} if opt_matches else None
            
            # Extract question text (before first option)
            if opt_matches:
                first_opt_pos = block.find(opt_matches[0][0])
                q_text = block[:first_opt_pos].strip()
            else:
                q_text = block.strip()
            
            if q_text:
                q_num += 1
                questions.append(ExamQuestion(
                    id=f"Q{q_num:04d}",
                    question=q_text,
                    options=options,
                    answer=None,  # Answers typically in separate key
                ))
            i += 2
        
        return questions


def load_exam_from_upload(uploader_widget) -> List[ExamQuestion]:
    """Loads and parses exam from the upload widget."""
    if not uploader_widget.value:
        raise ValueError("No file uploaded. Please upload an exam file first.")
    
    fname = list(uploader_widget.value.keys())[0]
    fdata = list(uploader_widget.value.values())[0]['content']
    ext = pathlib.Path(fname).suffix.lower()
    
    print(f"Parsing {fname} ...")
    if ext == ".json":
        questions = ExamParser.from_json(bytes(fdata))
    elif ext == ".pdf":
        questions = ExamParser.from_pdf(bytes(fdata))
    elif ext == ".docx":
        questions = ExamParser.from_docx(bytes(fdata))
    elif ext == ".txt":
        questions = ExamParser.from_txt(bytes(fdata))
    else:
        raise ValueError(f"Unsupported format: {ext}")
    
    return questions


def load_exam_from_path(file_path: str) -> List[ExamQuestion]:
    """Alternative: load exam from a local file path."""
    p = pathlib.Path(file_path)
    data = p.read_bytes()
    ext = p.suffix.lower()
    print(f"Loading exam from {file_path} ...")
    if ext == ".json":   return ExamParser.from_json(data)
    elif ext == ".pdf":  return ExamParser.from_pdf(data)
    elif ext == ".docx": return ExamParser.from_docx(data)
    elif ext == ".txt":  return ExamParser.from_txt(data)
    else: raise ValueError(f"Unsupported format: {ext}")


print("✅ Exam parser ready.")
print("→ Run the next cell to load your uploaded exam, OR use load_exam_from_path() for a file path.")

In [ ]:
# ============================================================
# SECTION 2C: LOAD & INSPECT EXAM
# ============================================================

import pandas as pd

# ── OPTION A: Load from the upload widget above ────────────────────────────────
# exam_questions = load_exam_from_upload(exam_uploader)

# ── OPTION B: Load from a local file path ─────────────────────────────────────
# exam_questions = load_exam_from_path("/path/to/your/exam.json")

# ── OPTION C: Use sample built-in exam (for testing the pipeline) ─────────────
SAMPLE_EXAM = [
    ExamQuestion(
        id="Q0001",
        question="A 58-year-old male presents with progressive dyspnea and a CT scan shows bilateral lower lobe ground-glass opacities with subpleural honeycombing. What is the most likely diagnosis?",
        options={"A": "Usual Interstitial Pneumonia (UIP)", "B": "Non-Specific Interstitial Pneumonia (NSIP)",
                 "C": "Cryptogenic Organizing Pneumonia", "D": "Hypersensitivity Pneumonitis"},
        answer="A",
        explanation="Subpleural honeycombing with basal predominance on CT is the hallmark of UIP/IPF pattern.",
        category="Thoracic Radiology",
        difficulty="medium",
        source="ABR_sample"
    ),
    ExamQuestion(
        id="Q0002",
        question="Which finding on MRI is most specific for hepatocellular carcinoma in a cirrhotic liver?",
        options={"A": "T2 hyperintensity", "B": "Arterial enhancement with washout",
                 "C": "Capsule appearance", "D": "Restricted diffusion"},
        answer="B",
        explanation="Arterial phase hyperenhancement followed by washout (LI-RADS 5) is the hallmark of HCC.",
        category="Abdominal Radiology",
        difficulty="hard",
        source="ABR_sample"
    ),
    ExamQuestion(
        id="Q0003",
        question="A patient has a 2.5 cm mass in the right upper lobe with spiculated margins on CT. PET shows SUV of 8.2. What is the Fleischner Society recommendation?",
        options={"A": "6-month follow-up CT", "B": "Annual low-dose CT",
                 "C": "Tissue sampling / biopsy", "D": "No follow-up needed"},
        answer="C",
        explanation="Spiculated margins and high FDG avidity are highly suspicious for malignancy. Tissue sampling is indicated.",
        category="Thoracic Radiology",
        difficulty="hard",
        source="ABR_sample"
    ),
    ExamQuestion(
        id="Q0004",
        question="What is the radiation dose reduction advantage of iterative reconstruction over filtered back projection in CT?",
        options={"A": "10-20%", "B": "30-50%", "C": "60-80%", "D": "No significant difference"},
        answer="B",
        explanation="Iterative reconstruction algorithms typically allow 30-50% dose reduction while maintaining diagnostic image quality.",
        category="CT Physics",
        difficulty="easy",
        source="ABR_sample"
    ),
    ExamQuestion(
        id="Q0005",
        question="A 35-year-old female presents with sudden severe headache. Non-contrast CT of the head shows hyperdensity in the subarachnoid space. What is the next step?",
        options={"A": "MRI brain", "B": "Lumbar puncture", "C": "CT angiography", "D": "EEG"},
        answer="C",
        explanation="CT angiography is the next step to identify the source of subarachnoid hemorrhage, typically an aneurysm.",
        category="Neuroradiology",
        difficulty="medium",
        source="ABR_sample"
    ),
]

exam_questions = SAMPLE_EXAM  # ← CHANGE THIS to use your uploaded exam

# ── Apply max_questions limit ──────────────────────────────────────────────────
if PIPELINE_CONFIG["max_questions"]:
    exam_questions = exam_questions[:PIPELINE_CONFIG["max_questions"]]

# ── Display summary ────────────────────────────────────────────────────────────
print(f"\n✅ Exam loaded: {len(exam_questions)} questions")

df_exam = pd.DataFrame([
    {"ID": q.id, "Category": q.category, "Difficulty": q.difficulty, 
     "Has Answer": q.answer is not None, "Has Options": q.options is not None,
     "Question (preview)": q.question[:80] + "..."}
    for q in exam_questions
])
display(df_exam)

## Section 3: Upload Your RAG Model

Three ways to plug in your model:

1. **Upload a `.pt`/`.pth` file** — PyTorch checkpoint (custom model)
2. **HuggingFace model ID** — loads from hub
3. **Implement `CustomRAGModel`** — inherit the base class below (most flexible, recommended for hospital deployments)

In [ ]:
# ============================================================
# SECTION 3A: MODEL UPLOAD WIDGET
# ============================================================

model_uploader = widgets.FileUpload(
    accept='.pt,.pth,.bin,.safetensors,.onnx,.pkl,.zip',
    multiple=False,
    description='🤖 Upload Model',
    layout=widgets.Layout(width='400px')
)

model_status = widgets.Output()

def on_model_upload(change):
    with model_status:
        clear_output()
        if model_uploader.value:
            fname = list(model_uploader.value.keys())[0]
            fsize = len(list(model_uploader.value.values())[0]['content'])
            print(f"✅ Uploaded: {fname} ({fsize/1024/1024:.2f} MB)")

model_uploader.observe(on_model_upload, names='value')

display(HTML("<h4>Step 2: Upload your model checkpoint (optional — skip if using HuggingFace ID or custom class)</h4>"))
display(model_uploader, model_status)

In [ ]:
# ============================================================
# SECTION 3B: BASE RAG MODEL INTERFACE
# All models must implement this interface to work with the pipeline.
# ============================================================

from abc import ABC, abstractmethod
from typing import List, Dict, Any, Optional, Tuple
import time
import numpy as np

@dataclass
class RAGResponse:
    """Standardized response from any RAG model."""
    question_id: str
    question: str
    predicted_answer: str          # Final answer (e.g. "A" for MCQ or free text)
    retrieved_passages: List[str]  # Top-k retrieved context passages
    retrieved_scores: List[float]  # Similarity scores for each passage
    generation: str                # Full generated response / reasoning
    latency_ms: float              # Total inference time
    metadata: Dict[str, Any] = field(default_factory=dict)


class BaseRAGModel(ABC):
    """
    Abstract base class for all RAG models in this pipeline.
    Your colleague should implement this class with her model.
    """
    
    def __init__(self, model_name: str, config: Optional[Dict] = None):
        self.model_name = model_name
        self.config = config or {}
        self._is_loaded = False
    
    @abstractmethod
    def load(self) -> None:
        """
        Load model weights, indexes, etc.
        Called once before inference.
        """
        pass

    @abstractmethod
    def retrieve(self, query: str, top_k: int = 5) -> Tuple[List[str], List[float]]:
        """
        Retrieve top-k relevant passages for a query.
        Returns: (passages, scores)
        """
        pass

    @abstractmethod
    def generate(self, query: str, context: List[str]) -> str:
        """
        Generate an answer given query and retrieved context.
        For MCQ, should return the option letter (A/B/C/D).
        """
        pass

    def answer(self, question: ExamQuestion, top_k: int = 5) -> RAGResponse:
        """Full RAG pipeline: retrieve + generate. Override if needed."""
        if not self._is_loaded:
            raise RuntimeError("Model not loaded. Call .load() first.")
        
        start = time.time()
        passages, scores = self.retrieve(question.question, top_k=top_k)
        generation = self.generate(question.question, passages)
        latency = (time.time() - start) * 1000
        
        # Extract option letter if MCQ
        predicted = self._extract_option(generation, question.options)
        
        return RAGResponse(
            question_id=question.id,
            question=question.question,
            predicted_answer=predicted,
            retrieved_passages=passages,
            retrieved_scores=scores,
            generation=generation,
            latency_ms=latency,
        )
    
    def _extract_option(self, text: str, options: Optional[Dict]) -> str:
        """Extract MCQ option letter from generated text."""
        if not options:
            return text.strip()
        # Look for option letter at start of response
        for letter in ["A", "B", "C", "D", "E"]:
            if text.strip().upper().startswith(letter):
                return letter
        # Search anywhere in text
        for letter in ["A", "B", "C", "D", "E"]:
            if f"({letter})" in text.upper() or f"option {letter}" in text.upper():
                return letter
        return text.strip()[:10]  # fallback


print("✅ Base model interface defined.")

In [ ]:
# ============================================================
# SECTION 3C: BUILT-IN MODEL IMPLEMENTATIONS
# ============================================================

import torch

# ──────────────────────────────────────────────────────────────────────────────
# MODEL 1: HuggingFace RAG Model
# Use any encoder-decoder model from HuggingFace hub.
# ──────────────────────────────────────────────────────────────────────────────
class HuggingFaceRAGModel(BaseRAGModel):
    """
    RAG using a HuggingFace generator + FAISS dense retriever.
    """
    
    def __init__(self, 
                 generator_model_id: str = "google/flan-t5-base",
                 embedding_model_id: str = "sentence-transformers/all-MiniLM-L6-v2",
                 knowledge_base_texts: Optional[List[str]] = None,
                 checkpoint_path: Optional[str] = None,
                 config: Optional[Dict] = None):
        super().__init__(generator_model_id, config)
        self.generator_model_id = generator_model_id
        self.embedding_model_id = embedding_model_id
        self.knowledge_base_texts = knowledge_base_texts or []
        self.checkpoint_path = checkpoint_path
        self.faiss_index = None
        self.embedder = None
        self.generator = None
        self.tokenizer = None
    
    def load(self):
        from sentence_transformers import SentenceTransformer
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        import faiss
        
        print(f"Loading embedder: {self.embedding_model_id}...")
        self.embedder = SentenceTransformer(self.embedding_model_id)
        
        print(f"Loading generator: {self.generator_model_id}...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.generator_model_id)
        self.generator = AutoModelForSeq2SeqLM.from_pretrained(self.generator_model_id)
        
        # Load checkpoint weights if provided
        if self.checkpoint_path:
            print(f"Loading checkpoint: {self.checkpoint_path}...")
            state = torch.load(self.checkpoint_path, map_location="cpu")
            if isinstance(state, dict) and "model_state_dict" in state:
                self.generator.load_state_dict(state["model_state_dict"], strict=False)
            elif isinstance(state, dict):
                self.generator.load_state_dict(state, strict=False)
            print("✅ Checkpoint loaded.")
        
        self.generator.eval()
        
        # Build FAISS index from knowledge base
        if self.knowledge_base_texts:
            print(f"Building FAISS index over {len(self.knowledge_base_texts)} passages...")
            embeddings = self.embedder.encode(self.knowledge_base_texts, 
                                             show_progress_bar=True, 
                                             convert_to_numpy=True)
            dim = embeddings.shape[1]
            self.faiss_index = faiss.IndexFlatIP(dim)
            faiss.normalize_L2(embeddings)
            self.faiss_index.add(embeddings)
            print(f"✅ FAISS index built: {self.faiss_index.ntotal} vectors.")
        
        self._is_loaded = True
        print(f"✅ {self.model_name} loaded.")
    
    def retrieve(self, query: str, top_k: int = 5) -> Tuple[List[str], List[float]]:
        if self.faiss_index is None or not self.knowledge_base_texts:
            return [], []
        q_emb = self.embedder.encode([query], convert_to_numpy=True)
        import faiss as _faiss
        _faiss.normalize_L2(q_emb)
        scores, idxs = self.faiss_index.search(q_emb, top_k)
        passages = [self.knowledge_base_texts[i] for i in idxs[0] if i >= 0]
        return passages, scores[0].tolist()
    
    def generate(self, query: str, context: List[str]) -> str:
        ctx = " ".join(context[:3]) if context else "No context available."
        prompt = f"Context: {ctx}\n\nQuestion: {query}\n\nAnswer:"
        inputs = self.tokenizer(prompt, return_tensors="pt", 
                                max_length=1024, truncation=True)
        with torch.no_grad():
            out = self.generator.generate(
                **inputs, 
                max_new_tokens=256,
                num_beams=4,
                early_stopping=True
            )
        return self.tokenizer.decode(out[0], skip_special_tokens=True)


# ──────────────────────────────────────────────────────────────────────────────
# MODEL 2: Custom PyTorch Checkpoint Model
# Use when you have a custom .pt / .pth file from training.
# ──────────────────────────────────────────────────────────────────────────────
class CustomCheckpointRAGModel(BaseRAGModel):
    """
    Load a custom PyTorch .pt/.pth model uploaded by the user.
    You need to define your model architecture class below.
    """
    
    def __init__(self, 
                 checkpoint_path: str,
                 model_architecture = None,  # Pass your nn.Module class
                 embedding_model_id: str = "sentence-transformers/all-MiniLM-L6-v2",
                 knowledge_base_texts: Optional[List[str]] = None,
                 config: Optional[Dict] = None):
        super().__init__(f"CustomModel({checkpoint_path})", config)
        self.checkpoint_path = checkpoint_path
        self.model_architecture = model_architecture
        self.embedding_model_id = embedding_model_id
        self.knowledge_base_texts = knowledge_base_texts or []
        self.model = None
        self.embedder = None
        self.faiss_index = None
    
    def load(self):
        from sentence_transformers import SentenceTransformer
        import faiss
        
        # Load embedder
        self.embedder = SentenceTransformer(self.embedding_model_id)
        
        # Load PyTorch checkpoint
        checkpoint = torch.load(self.checkpoint_path, map_location="cpu")
        
        if self.model_architecture is not None:
            # You provided the architecture class
            self.model = self.model_architecture()
            state = checkpoint.get("model_state_dict", checkpoint)
            self.model.load_state_dict(state, strict=False)
            self.model.eval()
        else:
            # Try to load as full model (torch.save(model, path))
            if hasattr(checkpoint, 'eval'):
                self.model = checkpoint
                self.model.eval()
            else:
                print("⚠️  Could not infer model architecture. "
                      "Pass model_architecture= parameter.")
                self.model = None
        
        # Build FAISS
        if self.knowledge_base_texts:
            embeddings = self.embedder.encode(self.knowledge_base_texts,
                                             show_progress_bar=True,
                                             convert_to_numpy=True)
            dim = embeddings.shape[1]
            self.faiss_index = faiss.IndexFlatIP(dim)
            faiss.normalize_L2(embeddings)
            self.faiss_index.add(embeddings)
        
        self._is_loaded = True
        print(f"✅ Custom checkpoint loaded from {self.checkpoint_path}")
    
    def retrieve(self, query: str, top_k: int = 5) -> Tuple[List[str], List[float]]:
        if self.faiss_index is None or not self.knowledge_base_texts:
            return [], []
        q_emb = self.embedder.encode([query], convert_to_numpy=True)
        import faiss as _faiss
        _faiss.normalize_L2(q_emb)
        scores, idxs = self.faiss_index.search(q_emb, top_k)
        passages = [self.knowledge_base_texts[i] for i in idxs[0] if i >= 0]
        return passages, scores[0].tolist()
    
    def generate(self, query: str, context: List[str]) -> str:
        # ═══════════════════════════════════════════════════════════════
        # TODO: Replace this with your model's actual inference logic.
        # Your model takes context + query and produces a string answer.
        # ═══════════════════════════════════════════════════════════════
        if self.model is None:
            return "Model not available."
        raise NotImplementedError(
            "Implement generate() for your custom model. "
            "Input: query (str), context (list of str). "
            "Output: answer string (e.g. 'A' for MCQ)."
        )


# ──────────────────────────────────────────────────────────────────────────────
# MODEL 3: API-Based RAG (OpenAI / Claude)
# For testing against frontier models as baselines.
# ──────────────────────────────────────────────────────────────────────────────
class APIBasedRAGModel(BaseRAGModel):
    """
    Uses an API-hosted LLM (GPT-4, Claude, etc.) for generation.
    Useful as a baseline comparison.
    """
    
    def __init__(self,
                 provider: str = "openai",  # "openai" or "anthropic"
                 model_id: str = "gpt-4-turbo",
                 api_key: Optional[str] = None,
                 embedding_model_id: str = "sentence-transformers/all-MiniLM-L6-v2",
                 knowledge_base_texts: Optional[List[str]] = None,
                 config: Optional[Dict] = None):
        super().__init__(f"{provider}/{model_id}", config)
        self.provider = provider
        self.model_id = model_id
        self.api_key = api_key or os.environ.get(
            "OPENAI_API_KEY" if provider == "openai" else "ANTHROPIC_API_KEY", "")
        self.embedding_model_id = embedding_model_id
        self.knowledge_base_texts = knowledge_base_texts or []
        self.embedder = None
        self.faiss_index = None
        self.client = None
    
    def load(self):
        from sentence_transformers import SentenceTransformer
        import faiss
        
        self.embedder = SentenceTransformer(self.embedding_model_id)
        
        if self.provider == "openai":
            from openai import OpenAI
            self.client = OpenAI(api_key=self.api_key)
        elif self.provider == "anthropic":
            from anthropic import Anthropic
            self.client = Anthropic(api_key=self.api_key)
        
        if self.knowledge_base_texts:
            embeddings = self.embedder.encode(self.knowledge_base_texts,
                                             show_progress_bar=True,
                                             convert_to_numpy=True)
            dim = embeddings.shape[1]
            self.faiss_index = faiss.IndexFlatIP(dim)
            faiss.normalize_L2(embeddings)
            self.faiss_index.add(embeddings)
        
        self._is_loaded = True
        print(f"✅ API model loaded: {self.model_name}")
    
    def retrieve(self, query: str, top_k: int = 5) -> Tuple[List[str], List[float]]:
        if self.faiss_index is None:
            return [], []
        q_emb = self.embedder.encode([query], convert_to_numpy=True)
        import faiss as _faiss
        _faiss.normalize_L2(q_emb)
        scores, idxs = self.faiss_index.search(q_emb, top_k)
        passages = [self.knowledge_base_texts[i] for i in idxs[0] if i >= 0]
        return passages, scores[0].tolist()
    
    def generate(self, query: str, context: List[str]) -> str:
        ctx = "\n".join(context[:3]) if context else "No context."
        sys_prompt = """You are an expert physician and medical knowledge assistant. 
Answer the medical question based on the provided context. 
For MCQ questions, respond with ONLY the option letter (A, B, C, or D) followed by a brief explanation."""
        user_msg = f"Context:\n{ctx}\n\nQuestion: {query}"
        
        if self.provider == "openai":
            resp = self.client.chat.completions.create(
                model=self.model_id,
                messages=[{"role": "system", "content": sys_prompt},
                          {"role": "user", "content": user_msg}],
                max_tokens=512, temperature=0
            )
            return resp.choices[0].message.content
        elif self.provider == "anthropic":
            resp = self.client.messages.create(
                model=self.model_id,
                max_tokens=512,
                system=sys_prompt,
                messages=[{"role": "user", "content": user_msg}]
            )
            return resp.content[0].text


# ──────────────────────────────────────────────────────────────────────────────
# MODEL 4: TEMPLATE — Your colleague fills this in
# ──────────────────────────────────────────────────────────────────────────────
class YourCustomRAGModel(BaseRAGModel):
    """
    ═══════════════════════════════════════════════════════════
    TEMPLATE FOR YOUR COLLEAGUE TO FILL IN.
    
    1. Implement load() — initialize your model
    2. Implement retrieve() — return (passages, scores)
    3. Implement generate() — return answer string
    ═══════════════════════════════════════════════════════════
    """
    
    def __init__(self, config: Optional[Dict] = None):
        super().__init__("MyCustomRAGModel", config)
        # ── Initialize your model attributes here ──────────────────────
        # self.my_model = None
        # self.my_retriever = None
    
    def load(self) -> None:
        # ── Load your model weights, indexes, etc. ──────────────────────
        # Example:
        # self.my_model = YourModelClass.from_pretrained("path/to/model")
        # self.my_retriever = YourRetriever(index_path="path/to/index")
        self._is_loaded = True
        print("✅ YourCustomRAGModel loaded.")
    
    def retrieve(self, query: str, top_k: int = 5) -> Tuple[List[str], List[float]]:
        # ── Return top_k relevant passages and their similarity scores ──
        # passages = self.my_retriever.search(query, top_k=top_k)
        # scores = [p.score for p in passages]
        # texts = [p.text for p in passages]
        # return texts, scores
        return ["Placeholder passage 1", "Placeholder passage 2"], [0.9, 0.8]
    
    def generate(self, query: str, context: List[str]) -> str:
        # ── Generate answer given query and context ─────────────────────
        # answer = self.my_model.generate(query=query, context=context)
        # return answer
        return "A"  # placeholder


print("✅ Model classes defined.")
print("\nAvailable model classes:")
print("  1. HuggingFaceRAGModel(generator_model_id, embedding_model_id, knowledge_base_texts)")
print("  2. CustomCheckpointRAGModel(checkpoint_path, model_architecture, knowledge_base_texts)")
print("  3. APIBasedRAGModel(provider, model_id, api_key, knowledge_base_texts)")
print("  4. YourCustomRAGModel — TEMPLATE: fill in your own logic")

In [ ]:
# ============================================================
# SECTION 3D: INSTANTIATE YOUR MODEL(S) FOR COMPARISON
# ============================================================
# Add as many models as you want to compare.
# Each entry: ("display_label", model_instance)

# ── Knowledge Base Setup ───────────────────────────────────────────────────────
# Paste your medical reference texts here OR load from a file.
# This is the corpus your RAG retrieves from.
KNOWLEDGE_BASE = [
    # EXAMPLE ENTRIES — Replace with your actual medical knowledge base
    "Usual Interstitial Pneumonia (UIP) is characterized by basal-predominant subpleural honeycombing with traction bronchiectasis on HRCT, representing the typical CT pattern of Idiopathic Pulmonary Fibrosis (IPF).",
    "Hepatocellular carcinoma (HCC) is diagnosed on MRI by the hallmark pattern of arterial phase hyperenhancement followed by portal venous or delayed phase washout appearance, classified as LI-RADS 5.",
    "The Fleischner Society guidelines recommend tissue sampling for solid pulmonary nodules >8mm with high-risk features including spiculated margins, upper lobe location, or high FDG avidity on PET.",
    "Iterative reconstruction algorithms in CT imaging allow a 30-50% reduction in radiation dose compared to conventional filtered back projection while maintaining diagnostic image quality.",
    "Subarachnoid hemorrhage on non-contrast CT appears as hyperdensity in the basal cisterns and sulci. CT angiography is the next step to identify the source, typically a cerebral aneurysm.",
    "The BI-RADS classification system for mammography: Category 0 (incomplete), 1 (negative), 2 (benign), 3 (probably benign, <2% malignancy), 4 (suspicious, biopsy recommended), 5 (highly suggestive of malignancy), 6 (known biopsy-proven malignancy).",
    "LI-RADS v2018 categorizes hepatic observations in patients at risk for HCC: LR-1 (definitely benign), LR-2 (probably benign), LR-3 (intermediate), LR-4 (probably HCC), LR-5 (definitely HCC).",
    "In NEET UG, a single best response MCQ format is used. Each correct answer carries +4 marks and each incorrect answer carries -1 mark (negative marking).",
]

# ── Load knowledge base from a text file (optional) ───────────────────────────
# with open("/path/to/knowledge_base.txt") as f:
#     KNOWLEDGE_BASE = [line.strip() for line in f if line.strip()]

# ── DEFINE MODELS TO COMPARE ──────────────────────────────────────────────────
MODELS_TO_EVALUATE = [
    # ── Model 1: Lightweight HuggingFace baseline
    ("flan-t5-base (baseline)", 
     HuggingFaceRAGModel(
         generator_model_id="google/flan-t5-base",
         embedding_model_id=RETRIEVAL_CONFIG["embedding_model"],
         knowledge_base_texts=KNOWLEDGE_BASE,
     )
    ),
    
    # ── Model 2: Custom PyTorch checkpoint (uncomment to use)
    # ("My Custom Model v2",
    #  CustomCheckpointRAGModel(
    #      checkpoint_path="/path/to/model.pt",
    #      knowledge_base_texts=KNOWLEDGE_BASE,
    #  )
    # ),
    
    # ── Model 3: Your colleague's model (uncomment & fill in YourCustomRAGModel)
    # ("Colleague Model",
    #  YourCustomRAGModel()
    # ),
    
    # ── Model 4: API baseline (requires API key)
    # ("GPT-4 Turbo (API baseline)",
    #  APIBasedRAGModel(
    #      provider="openai",
    #      model_id="gpt-4-turbo",
    #      api_key=JUDGE_CONFIG["openai_api_key"],
    #      knowledge_base_texts=KNOWLEDGE_BASE,
    #  )
    # ),
]

# ── Load all models ────────────────────────────────────────────────────────────
print(f"Loading {len(MODELS_TO_EVALUATE)} model(s)...\n")
for label, model in MODELS_TO_EVALUATE:
    print(f"Loading: {label}")
    model.load()
    print()

print(f"\n✅ All models loaded. Ready for evaluation.")

## Section 4: Run Evaluation

In [ ]:
# ============================================================
# SECTION 4A: METRICS ENGINE
# Computes all configured metrics per response.
# ============================================================

from rouge_score import rouge_scorer
from tqdm.notebook import tqdm
import nltk
nltk.download('punkt', quiet=True)

@dataclass
class QuestionResult:
    question_id: str
    question: str
    ground_truth: Optional[str]
    predicted_answer: str
    generation: str
    retrieved_passages: List[str]
    retrieved_scores: List[float]
    latency_ms: float
    exact_match: Optional[bool] = None
    rouge1: Optional[float] = None
    rouge2: Optional[float] = None
    rougeL: Optional[float] = None
    bert_score_f1: Optional[float] = None
    retrieval_precision: Optional[float] = None
    faithfulness_score: Optional[float] = None
    llm_judge_score: Optional[float] = None
    llm_judge_rationale: Optional[str] = None
    category: Optional[str] = None
    difficulty: Optional[str] = None
    error: Optional[str] = None


class MedicalRAGEvaluator:
    """Evaluates RAG responses using multiple metrics."""
    
    def __init__(self, metrics_config: Dict, judge_config: Dict):
        self.cfg = metrics_config
        self.judge_cfg = judge_config
        self._rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
        self._judge_client = None
        if metrics_config.get("llm_judge") and judge_config["judge_model"] != "none":
            self._init_judge()
    
    def _init_judge(self):
        jm = self.judge_cfg["judge_model"]
        if jm.startswith("gpt"):
            from openai import OpenAI
            self._judge_client = ("openai", OpenAI(api_key=self.judge_cfg["openai_api_key"]), jm)
        elif jm.startswith("claude"):
            from anthropic import Anthropic
            self._judge_client = ("anthropic", Anthropic(api_key=self.judge_cfg["anthropic_api_key"]), jm)
        print(f"✅ LLM Judge initialized: {jm}")
    
    def compute_exact_match(self, pred: str, gt: str) -> bool:
        return pred.strip().upper() == gt.strip().upper()
    
    def compute_rouge(self, pred: str, gt: str) -> Dict[str, float]:
        scores = self._rouge.score(gt, pred)
        return {"rouge1": scores["rouge1"].fmeasure,
                "rouge2": scores["rouge2"].fmeasure,
                "rougeL": scores["rougeL"].fmeasure}
    
    def compute_bert_score(self, pred: str, gt: str) -> float:
        try:
            from bert_score import score as bert_score
            P, R, F = bert_score([pred], [gt], lang="en", verbose=False, 
                                  model_type="distilbert-base-uncased")
            return float(F[0])
        except Exception:
            return 0.0
    
    def compute_retrieval_precision(self, passages: List[str], ground_truth: str,
                                     question: str) -> float:
        """
        Heuristic: passage is 'relevant' if it contains key terms from the
        ground truth answer or question domain.
        For production: use human-annotated relevance labels.
        """
        if not passages or not ground_truth:
            return 0.0
        gt_terms = set(ground_truth.lower().split())
        relevant_count = sum(
            1 for p in passages 
            if len(set(p.lower().split()) & gt_terms) > 0
        )
        return relevant_count / len(passages)
    
    def compute_faithfulness(self, generation: str, passages: List[str]) -> float:
        """
        Simple faithfulness: fraction of non-trivial tokens in generation
        that appear in retrieved context.
        Production: use NLI model (e.g. MedNLI) for proper faithfulness.
        """
        if not passages or not generation:
            return 0.0
        ctx = " ".join(passages).lower()
        stop_words = {"the", "a", "an", "is", "in", "of", "and", "or", "to", "for"}
        gen_tokens = [t for t in generation.lower().split() if t not in stop_words and len(t) > 3]
        if not gen_tokens:
            return 0.0
        grounded = sum(1 for t in gen_tokens if t in ctx)
        return grounded / len(gen_tokens)
    
    def llm_judge_score(self, question: str, gt: str, pred_generation: str,
                         options: Optional[Dict]) -> Tuple[float, str]:
        """LLM-as-judge for clinical accuracy (GPQA Diamond style)."""
        if self._judge_client is None:
            return None, None
        
        opts_str = json.dumps(options) if options else ""
        prompt = f"""You are evaluating a medical AI system on a professional exam question.

Question: {question}
Options: {opts_str}
Correct Answer: {gt}
AI Response: {pred_generation}

Score the AI response from 0-10 based on:
- Clinical accuracy (0-4 points)
- Reasoning quality (0-3 points)
- Safety and appropriateness for clinical use (0-3 points)

Respond in JSON: {{"score": <0-10>, "rationale": "<1-2 sentences>"}}"""        
        try:
            provider, client, model_id = self._judge_client
            if provider == "openai":
                resp = client.chat.completions.create(
                    model=model_id,
                    messages=[{"role": "system", "content": self.judge_cfg["judge_system_prompt"]},
                              {"role": "user", "content": prompt}],
                    temperature=0, max_tokens=200
                )
                result = json.loads(resp.choices[0].message.content)
            elif provider == "anthropic":
                resp = client.messages.create(
                    model=model_id, max_tokens=200,
                    system=self.judge_cfg["judge_system_prompt"],
                    messages=[{"role": "user", "content": prompt}]
                )
                result = json.loads(resp.content[0].text)
            return float(result["score"]) / 10.0, result.get("rationale", "")
        except Exception as e:
            return None, str(e)

    def evaluate_response(self, response: RAGResponse, question: ExamQuestion) -> QuestionResult:
        gt = question.answer
        pred = response.predicted_answer
        gen = response.generation
        
        result = QuestionResult(
            question_id=question.id,
            question=question.question,
            ground_truth=gt,
            predicted_answer=pred,
            generation=gen,
            retrieved_passages=response.retrieved_passages,
            retrieved_scores=response.retrieved_scores,
            latency_ms=response.latency_ms,
            category=question.category,
            difficulty=question.difficulty,
        )
        
        if gt:
            if self.cfg.get("exact_match"):
                result.exact_match = self.compute_exact_match(pred, gt)
            
            if self.cfg.get("rouge"):
                rouge = self.compute_rouge(gen, gt)
                result.rouge1 = rouge["rouge1"]
                result.rouge2 = rouge["rouge2"]
                result.rougeL = rouge["rougeL"]
            
            if self.cfg.get("bert_score"):
                result.bert_score_f1 = self.compute_bert_score(gen, gt)
            
            if self.cfg.get("retrieval_precision"):
                result.retrieval_precision = self.compute_retrieval_precision(
                    response.retrieved_passages, gt, question.question
                )
            
            if self.cfg.get("llm_judge") and self._judge_client:
                s, r = self.llm_judge_score(question.question, gt, gen, question.options)
                result.llm_judge_score = s
                result.llm_judge_rationale = r
        
        if self.cfg.get("hallucination_score"):
            result.faithfulness_score = self.compute_faithfulness(
                gen, response.retrieved_passages
            )
        
        return result


evaluator = MedicalRAGEvaluator(METRICS_CONFIG, JUDGE_CONFIG)
print("✅ Evaluator ready.")

In [ ]:
# ============================================================
# SECTION 4B: RUN FULL EVALUATION LOOP
# ============================================================

all_model_results: Dict[str, List[QuestionResult]] = {}

for model_label, model in MODELS_TO_EVALUATE:
    print(f"\n{'='*60}")
    print(f"🔬 Evaluating: {model_label}")
    print(f"{'='*60}")
    
    results = []
    for question in tqdm(exam_questions, desc=model_label):
        try:
            response = model.answer(
                question, 
                top_k=RETRIEVAL_CONFIG["top_k"]
            )
            result = evaluator.evaluate_response(response, question)
        except Exception as e:
            result = QuestionResult(
                question_id=question.id,
                question=question.question,
                ground_truth=question.answer,
                predicted_answer="ERROR",
                generation="",
                retrieved_passages=[],
                retrieved_scores=[],
                latency_ms=0,
                error=str(e),
                category=question.category,
                difficulty=question.difficulty,
            )
            print(f"  ⚠️  Error on {question.id}: {e}")
        results.append(result)
    
    all_model_results[model_label] = results
    
    # Quick accuracy printout
    answered = [r for r in results if r.ground_truth and r.exact_match is not None]
    if answered:
        acc = np.mean([r.exact_match for r in answered])
        print(f"\n✅ {model_label} — Accuracy: {acc:.1%} ({sum(r.exact_match for r in answered)}/{len(answered)})")

print(f"\n\n✅ Evaluation complete for all {len(MODELS_TO_EVALUATE)} model(s).")

## Section 5: Results & Analysis

In [ ]:
# ============================================================
# SECTION 5A: AGGREGATE METRICS SUMMARY
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

def compute_aggregate(results: List[QuestionResult]) -> Dict:
    def safe_mean(vals):
        vals = [v for v in vals if v is not None]
        return float(np.mean(vals)) if vals else None
    
    answered = [r for r in results if r.ground_truth]
    errors = [r for r in results if r.error]
    
    return {
        "n_questions": len(results),
        "n_with_gt": len(answered),
        "n_errors": len(errors),
        "accuracy": safe_mean([r.exact_match for r in answered]),
        "rouge1": safe_mean([r.rouge1 for r in answered]),
        "rouge2": safe_mean([r.rouge2 for r in answered]),
        "rougeL": safe_mean([r.rougeL for r in answered]),
        "bert_score_f1": safe_mean([r.bert_score_f1 for r in answered]),
        "retrieval_precision": safe_mean([r.retrieval_precision for r in answered]),
        "faithfulness": safe_mean([r.faithfulness_score for r in results]),
        "avg_latency_ms": safe_mean([r.latency_ms for r in results]),
        "p50_latency_ms": float(np.percentile([r.latency_ms for r in results], 50)) if results else None,
        "p95_latency_ms": float(np.percentile([r.latency_ms for r in results], 95)) if results else None,
        "llm_judge_score": safe_mean([r.llm_judge_score for r in results]),
    }

# Build summary table
summary_rows = []
for label, results in all_model_results.items():
    agg = compute_aggregate(results)
    agg["model"] = label
    summary_rows.append(agg)

df_summary = pd.DataFrame(summary_rows).set_index("model")

# Format for display
display_cols = ["accuracy", "rouge1", "rouge2", "rougeL", "bert_score_f1",
                "retrieval_precision", "faithfulness", "avg_latency_ms"]
df_display = df_summary[display_cols].copy()
for col in display_cols:
    if "latency" in col:
        df_display[col] = df_display[col].apply(lambda x: f"{x:.1f}ms" if x else "N/A")
    else:
        df_display[col] = df_display[col].apply(lambda x: f"{x:.3f}" if x is not None else "N/A")

print(f"\n{'='*70}")
print(f"📊 EVALUATION SUMMARY — {PIPELINE_CONFIG['exam_type']} | Run: {RUN_ID}")
print(f"{'='*70}")
display(df_display)

# Save CSV
df_summary.to_csv(RUN_DIR / "summary.csv")
print(f"\n💾 Summary saved to {RUN_DIR / 'summary.csv'}")

In [ ]:
# ============================================================
# SECTION 5B: VISUALIZATIONS
# ============================================================

sns.set_style("whitegrid")
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f"Medical RAG Evaluation — {PIPELINE_CONFIG['exam_type']}\nRun: {RUN_ID}", 
             fontsize=14, fontweight='bold')

model_labels = list(all_model_results.keys())
colors = sns.color_palette("husl", len(model_labels))

# ── Plot 1: Accuracy ──────────────────────────────────────────────────────────
ax = axes[0, 0]
acc_vals = [compute_aggregate(all_model_results[m])["accuracy"] or 0 for m in model_labels]
bars = ax.bar(range(len(model_labels)), acc_vals, color=colors)
ax.set_xticks(range(len(model_labels)))
ax.set_xticklabels(model_labels, rotation=20, ha='right', fontsize=8)
ax.set_ylim(0, 1)
ax.set_ylabel("Accuracy")
ax.set_title("Exact Match Accuracy")
for bar, val in zip(bars, acc_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f"{val:.1%}", ha='center', fontsize=9, fontweight='bold')

# ── Plot 2: ROUGE Scores ──────────────────────────────────────────────────────
ax = axes[0, 1]
x = np.arange(len(model_labels))
w = 0.25
for i, (rk, rn) in enumerate([("rouge1","R1"),("rouge2","R2"),("rougeL","RL")]):
    vals = [compute_aggregate(all_model_results[m])[rk] or 0 for m in model_labels]
    ax.bar(x + i*w, vals, w, label=rn)
ax.set_xticks(x + w)
ax.set_xticklabels(model_labels, rotation=20, ha='right', fontsize=8)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("ROUGE Scores")
ax.legend()

# ── Plot 3: BERTScore ─────────────────────────────────────────────────────────
ax = axes[0, 2]
bs_vals = [compute_aggregate(all_model_results[m])["bert_score_f1"] or 0 for m in model_labels]
bars = ax.bar(range(len(model_labels)), bs_vals, color=colors)
ax.set_xticks(range(len(model_labels)))
ax.set_xticklabels(model_labels, rotation=20, ha='right', fontsize=8)
ax.set_ylim(0, 1)
ax.set_ylabel("BERTScore F1")
ax.set_title("BERTScore (Semantic Similarity)")

# ── Plot 4: Faithfulness / Hallucination ─────────────────────────────────────
ax = axes[1, 0]
f_vals = [compute_aggregate(all_model_results[m])["faithfulness"] or 0 for m in model_labels]
bars = ax.bar(range(len(model_labels)), f_vals, color=colors)
ax.set_xticks(range(len(model_labels)))
ax.set_xticklabels(model_labels, rotation=20, ha='right', fontsize=8)
ax.set_ylim(0, 1)
ax.set_ylabel("Faithfulness")
ax.set_title("Faithfulness Score\n(Higher = Less Hallucination)")

# ── Plot 5: Retrieval Precision ───────────────────────────────────────────────
ax = axes[1, 1]
rp_vals = [compute_aggregate(all_model_results[m])["retrieval_precision"] or 0 for m in model_labels]
bars = ax.bar(range(len(model_labels)), rp_vals, color=colors)
ax.set_xticks(range(len(model_labels)))
ax.set_xticklabels(model_labels, rotation=20, ha='right', fontsize=8)
ax.set_ylim(0, 1)
ax.set_ylabel("Precision@k")
ax.set_title("Retrieval Precision@k")

# ── Plot 6: Latency Distribution ──────────────────────────────────────────────
ax = axes[1, 2]
for i, (label, results) in enumerate(all_model_results.items()):
    latencies = [r.latency_ms for r in results if r.latency_ms > 0]
    if latencies:
        ax.hist(latencies, bins=10, alpha=0.7, label=label, color=colors[i])
ax.set_xlabel("Latency (ms)")
ax.set_ylabel("Count")
ax.set_title("Latency Distribution")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(RUN_DIR / "metrics_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"💾 Plot saved to {RUN_DIR / 'metrics_overview.png'}")

In [ ]:
# ============================================================
# SECTION 5C: PERFORMANCE BY CATEGORY & DIFFICULTY
# ============================================================

for model_label, results in all_model_results.items():
    answered = [r for r in results if r.ground_truth and r.exact_match is not None]
    if not answered:
        continue
    
    df_r = pd.DataFrame([
        {"category": r.category or "Unknown",
         "difficulty": r.difficulty or "Unknown",
         "correct": int(r.exact_match),
         "latency_ms": r.latency_ms,
         "faithfulness": r.faithfulness_score}
        for r in answered
    ])
    
    print(f"\n{'─'*50}")
    print(f"📋 Model: {model_label}")
    print(f"{'─'*50}")
    
    # By category
    if df_r["category"].nunique() > 1:
        cat_acc = df_r.groupby("category")["correct"].agg(["mean", "count"]).round(3)
        cat_acc.columns = ["Accuracy", "N"]
        cat_acc = cat_acc.sort_values("Accuracy", ascending=False)
        print("\n📂 Accuracy by Category:")
        display(cat_acc)
    
    # By difficulty
    if df_r["difficulty"].nunique() > 1:
        diff_order = ["easy", "medium", "hard"]
        diff_acc = df_r.groupby("difficulty")["correct"].agg(["mean", "count"]).round(3)
        diff_acc.columns = ["Accuracy", "N"]
        print("\n🎯 Accuracy by Difficulty:")
        display(diff_acc)

In [ ]:
# ============================================================
# SECTION 5D: GPQA DIAMOND STYLE — ERROR ANALYSIS
# ============================================================
# Shows wrong answers with retrieved context for manual inspection.

print("\n🔍 GPQA Diamond Style — Wrong Answer Analysis")
print("(Double-blind evaluation: see what context the model had)\n")

for model_label, results in all_model_results.items():
    wrong = [r for r in results 
             if r.ground_truth and r.exact_match is not None and not r.exact_match]
    print(f"\n{'='*60}")
    print(f"Model: {model_label} — {len(wrong)} wrong answers")
    print(f"{'='*60}")
    
    for r in wrong[:5]:  # Show max 5 examples
        print(f"\n[{r.question_id}] Category: {r.category} | Difficulty: {r.difficulty}")
        print(f"❓ Q: {r.question[:150]}...")
        print(f"✅ Ground Truth: {r.ground_truth}")
        print(f"❌ Model Answer: {r.predicted_answer}")
        print(f"💬 Generation: {r.generation[:200]}...")
        print(f"📎 Retrieved ({len(r.retrieved_passages)} passages):")
        for i, p in enumerate(r.retrieved_passages[:2]):
            print(f"   [{i+1}] {p[:120]}...")
        print(f"⏱️  Latency: {r.latency_ms:.0f}ms")

In [ ]:
# ============================================================
# SECTION 5E: SAVE DETAILED RESULTS (JSON + CSV)
# ============================================================

from dataclasses import asdict

all_results_dict = {}
for model_label, results in all_model_results.items():
    all_results_dict[model_label] = [asdict(r) for r in results]

# Save JSON
json_path = RUN_DIR / "detailed_results.json"
with open(json_path, "w") as f:
    json.dump({
        "run_id": RUN_ID,
        "config": {
            "pipeline": PIPELINE_CONFIG,
            "retrieval": RETRIEVAL_CONFIG,
            "metrics": METRICS_CONFIG,
        },
        "summary": df_summary.to_dict(),
        "results": all_results_dict
    }, f, indent=2, default=str)
print(f"💾 Detailed results saved to {json_path}")

# Save per-model CSVs
for model_label, results in all_model_results.items():
    safe_name = re.sub(r'[^\w]', '_', model_label)[:40]
    csv_path = RUN_DIR / f"results_{safe_name}.csv"
    df_r = pd.DataFrame([asdict(r) for r in results])
    df_r.drop(columns=["retrieved_passages", "retrieved_scores"], 
              errors="ignore").to_csv(csv_path, index=False)
    print(f"💾 CSV saved: {csv_path.name}")

print(f"\n✅ All results saved to: {RUN_DIR.resolve()}")

In [ ]:
# ============================================================
# SECTION 5F: GENERATE HTML EVALUATION REPORT
# ============================================================

def generate_html_report(summary_df, all_results, config, run_id, output_path):
    from jinja2 import Template
    
    html_template = """
<!DOCTYPE html>
<html>
<head>
<title>Medical RAG Evaluation Report</title>
<style>
  body { font-family: Arial, sans-serif; margin: 40px; color: #222; }
  h1 { color: #1a3a5c; border-bottom: 3px solid #1a3a5c; padding-bottom: 10px; }
  h2 { color: #2c6496; margin-top: 30px; }
  h3 { color: #3a7abf; }
  .badge { display: inline-block; padding: 3px 10px; border-radius: 12px;
           font-size: 12px; font-weight: bold; }
  .badge-green { background: #d4edda; color: #155724; }
  .badge-blue { background: #cce5ff; color: #004085; }
  .badge-red { background: #f8d7da; color: #721c24; }
  table { border-collapse: collapse; width: 100%; margin: 15px 0; }
  th { background: #1a3a5c; color: white; padding: 10px; text-align: left; }
  td { padding: 8px 10px; border-bottom: 1px solid #ddd; }
  tr:nth-child(even) { background: #f8f9fa; }
  .highlight { background: #fff3cd !important; }
  .metric-card { display: inline-block; margin: 8px; padding: 15px 25px;
                 border-radius: 8px; background: #f0f4f8; border-left: 4px solid #2c6496; }
  .metric-val { font-size: 28px; font-weight: bold; color: #1a3a5c; }
  .metric-name { font-size: 12px; color: #666; }
  .error-box { background: #fff3f3; border-left: 4px solid #dc3545;
               padding: 10px 15px; margin: 5px 0; border-radius: 4px; }
  .warn { color: #856404; background: #fff3cd; padding: 2px 8px; border-radius: 4px; }
  footer { margin-top: 40px; border-top: 1px solid #ddd; padding-top: 10px;
           font-size: 12px; color: #888; }
</style>
</head>
<body>
<h1>🏥 Medical RAG Evaluation Report</h1>

<p>
  <span class="badge badge-blue">{{ config.pipeline.exam_type }}</span>&nbsp;
  <span class="badge badge-green">Run: {{ run_id }}</span>&nbsp;
  <span class="badge badge-blue">{{ config.pipeline.institution }}</span>
  <br><small>Evaluator: {{ config.pipeline.evaluator_name }} &nbsp;|&nbsp;
  Generated: {{ timestamp }}</small>
</p>

<h2>📊 Summary</h2>
<table>
  <tr>
    <th>Model</th>
    <th>Accuracy</th>
    <th>ROUGE-1</th>
    <th>ROUGE-L</th>
    <th>BERTScore F1</th>
    <th>Retrieval Precision</th>
    <th>Faithfulness</th>
    <th>Avg Latency (ms)</th>
    <th>N</th>
  </tr>
  {% for model, row in summary.items() %}
  <tr>
    <td><strong>{{ model }}</strong></td>
    <td>{{ '%.1f%%' % (row.accuracy * 100) if row.accuracy else 'N/A' }}</td>
    <td>{{ '%.3f' % row.rouge1 if row.rouge1 else 'N/A' }}</td>
    <td>{{ '%.3f' % row.rougeL if row.rougeL else 'N/A' }}</td>
    <td>{{ '%.3f' % row.bert_score_f1 if row.bert_score_f1 else 'N/A' }}</td>
    <td>{{ '%.3f' % row.retrieval_precision if row.retrieval_precision else 'N/A' }}</td>
    <td>{{ '%.3f' % row.faithfulness if row.faithfulness else 'N/A' }}</td>
    <td>{{ '%.0f' % row.avg_latency_ms if row.avg_latency_ms else 'N/A' }}</td>
    <td>{{ row.n_questions }}</td>
  </tr>
  {% endfor %}
</table>

<h2>🔍 Wrong Answer Analysis (Sample)</h2>
{% for model_label, results in all_results.items() %}
<h3>{{ model_label }}</h3>
{% set wrong = [] %}
{% for r in results %}
  {% if r.ground_truth and r.exact_match == false %}
    {% if wrong.append(r) %}{% endif %}
  {% endif %}
{% endfor %}
<p>Wrong answers: <strong>{{ wrong|length }}</strong> / {{ results|length }}</p>
{% for r in wrong[:3] %}
<div class="error-box">
  <strong>[{{ r.question_id }}]</strong> {{ r.category }} | {{ r.difficulty }}<br>
  <em>Q: {{ r.question[:200] }}...</em><br>
  ✅ Correct: <strong>{{ r.ground_truth }}</strong> &nbsp;
  ❌ Predicted: <strong class="warn">{{ r.predicted_answer }}</strong><br>
  💬 {{ r.generation[:150] }}...
</div>
{% endfor %}
{% endfor %}

<footer>
Medical RAG Testing Pipeline | Industry-Grade Hospital AI Evaluation Framework<br>
All results saved at: {{ output_path }}
</footer>
</body></html>
"""
    
    summary_data = {}
    for model_label, results in all_results.items():
        agg = compute_aggregate(results)
        summary_data[model_label] = type('Row', (), agg)()
    
    t = Template(html_template)
    html = t.render(
        config=type('C', (), {
            'pipeline': type('P', (), PIPELINE_CONFIG)()
        })(),
        run_id=run_id,
        timestamp=datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        summary=summary_data,
        all_results={k: [type('R', (), r)() for r in v] 
                     for k, v in all_results_dict.items()},
        output_path=str(output_path)
    )
    
    html_path = output_path / "evaluation_report.html"
    html_path.write_text(html)
    return html_path


report_path = generate_html_report(
    df_summary, all_model_results, 
    {"pipeline": PIPELINE_CONFIG}, 
    RUN_ID, RUN_DIR
)
print(f"\n📄 HTML Report saved: {report_path}")
print(f"\n🎉 EVALUATION COMPLETE!")
print(f"📁 All outputs saved to: {RUN_DIR.resolve()}")
print(f"   ├── summary.csv")
print(f"   ├── detailed_results.json")
print(f"   ├── metrics_overview.png")
print(f"   ├── results_<model>.csv (one per model)")
print(f"   └── evaluation_report.html")

## Section 6: Multi-Run Comparison

Compare results across **multiple pipeline runs** (different days, different models).

In [ ]:
# ============================================================
# SECTION 6: COMPARE ACROSS RUNS
# Load previous run summaries and compare.
# ============================================================

def load_all_run_summaries(results_dir: str = "./rag_eval_results") -> pd.DataFrame:
    """
    Scans the output directory for all previous run summaries
    and combines them into a single comparison DataFrame.
    """
    base = pathlib.Path(results_dir)
    all_summaries = []
    
    for run_dir in sorted(base.iterdir()):
        summary_file = run_dir / "summary.csv"
        json_file = run_dir / "detailed_results.json"
        if summary_file.exists():
            df = pd.read_csv(summary_file)
            df["run_id"] = run_dir.name
            # Try to get exam_type from json
            if json_file.exists():
                try:
                    meta = json.loads(json_file.read_text())
                    df["exam_type"] = meta["config"]["pipeline"].get("exam_type", "unknown")
                except:
                    df["exam_type"] = "unknown"
            all_summaries.append(df)
    
    if not all_summaries:
        print("No previous runs found.")
        return pd.DataFrame()
    
    return pd.concat(all_summaries, ignore_index=True)


df_all_runs = load_all_run_summaries(PIPELINE_CONFIG["output_dir"])

if not df_all_runs.empty:
    print(f"\n📈 Cross-Run Comparison ({len(df_all_runs)} total model evaluations)\n")
    display_cols = ["run_id", "exam_type", "model", "accuracy", "rouge1", 
                    "bert_score_f1", "faithfulness", "avg_latency_ms", "n_questions"]
    display_cols = [c for c in display_cols if c in df_all_runs.columns]
    display(df_all_runs[display_cols].sort_values(["exam_type", "accuracy"], ascending=[True, False]))
    
    # Plot accuracy trend
    if "accuracy" in df_all_runs.columns and "model" in df_all_runs.columns:
        fig, ax = plt.subplots(figsize=(12, 5))
        for model_grp, grp_df in df_all_runs.groupby("model"):
            ax.plot(grp_df["run_id"], grp_df["accuracy"], marker='o', label=model_grp)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
        ax.set_ylabel("Accuracy")
        ax.set_title("Model Accuracy Across Runs")
        ax.legend()
        plt.tight_layout()
        plt.show()
else:
    print("Only one run exists so far. Run the pipeline multiple times to compare.")

## Section 7: Knowledge Base Builder

Build a FAISS knowledge base from your medical PDFs.

In [ ]:
# ============================================================
# SECTION 7: BUILD KNOWLEDGE BASE FROM PDF CORPUS
# Upload your medical textbooks / guidelines / reference docs.
# ============================================================

kb_uploader = widgets.FileUpload(
    accept='.pdf,.txt,.docx',
    multiple=True,
    description='📚 Upload KB Docs',
    layout=widgets.Layout(width='400px')
)

display(HTML("<h4>Upload medical reference documents (textbooks, guidelines, etc.)</h4>"
             "<p>These form the knowledge base your RAG model retrieves from.</p>"))
display(kb_uploader)


def build_knowledge_base_from_uploads(uploader, chunk_size=512, chunk_overlap=64):
    """Extracts and chunks text from uploaded documents."""
    from pypdf import PdfReader
    import re
    
    all_texts = []
    
    for fname, finfo in uploader.value.items():
        ext = pathlib.Path(fname).suffix.lower()
        data = bytes(finfo['content'])
        print(f"Processing: {fname}...")
        
        if ext == ".pdf":
            reader = PdfReader(io.BytesIO(data))
            for page in reader.pages:
                all_texts.append(page.extract_text() or "")
        elif ext == ".txt":
            all_texts.append(data.decode("utf-8", errors="ignore"))
        elif ext == ".docx":
            from docx import Document
            doc = Document(io.BytesIO(data))
            all_texts.append(" ".join(p.text for p in doc.paragraphs))
    
    # Chunk texts
    chunks = []
    for text in all_texts:
        text = re.sub(r'\s+', ' ', text).strip()
        words = text.split()
        for i in range(0, len(words), chunk_size - chunk_overlap):
            chunk = " ".join(words[i:i + chunk_size])
            if len(chunk.split()) > 20:  # skip tiny chunks
                chunks.append(chunk)
    
    print(f"\n✅ Knowledge base built: {len(chunks)} chunks from {len(uploader.value)} documents")
    return chunks


print("→ After uploading docs, run:")
print("  KNOWLEDGE_BASE = build_knowledge_base_from_uploads(kb_uploader)")
print("  Then reload your model(s) in Section 3D with the new KNOWLEDGE_BASE.")

---

## 🏁 Pipeline Complete

### Quick Reference for Your Colleague

To plug in **your own model**, do one of the following:

**Option A — Simplest:** Fill in `YourCustomRAGModel` in Section 3C:
```python
class YourCustomRAGModel(BaseRAGModel):
    def load(self): ...      # load your model
    def retrieve(self, query, top_k): ...  # return (passages, scores)
    def generate(self, query, context): ... # return answer string
```

**Option B — Upload `.pt`/`.pth`:** Use `CustomCheckpointRAGModel` in Section 3D.

**Option C — HuggingFace model:** Use `HuggingFaceRAGModel` with your model ID.

**Option D — API model:** Use `APIBasedRAGModel` with your OpenAI/Anthropic key.

### Output Files
| File | Description |
|------|-------------|
| `summary.csv` | Aggregate metrics per model |
| `detailed_results.json` | Full per-question results with retrieved passages |
| `results_<model>.csv` | Per-question CSV for each model |
| `metrics_overview.png` | Comparison charts |
| `evaluation_report.html` | Full HTML report |
